Strategy based on hand modelling of heat and mass balance

In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import math
import re
import json
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

In [2]:
#Experimental arrangement
bed_depth_m = 0.01
bed_radius_m = 0.025
bed_diameter_m = 2 * bed_radius_m

T0_K = 298.15  # Initial temperature in Kelvin
T_air_K = 343.15  # Air temperature in Kelvin

total_time = 240*60  # Total time in seconds

diameters = [610e-6, 450e-6, 236e-6, 213e-6]
velocities = [0.22, 0.48]

water_latent_heat = 2_261_000.0  # Latent heat of water in J/kg
product_density = 1081.0  # Solid density in kg/m3
product_cp = 600.0  # Solid specific heat capacity in J/(kg K)
product_k = 0.02  # Solid thermal conductivity in W/(m K)
R_J_mol_K = 8.31446261815324  # Universal gas constant in J/(mol K)
#Air properties at 70°C from https://www.engineeringtoolbox.com
air_dynamic_viscosity = 20.44e-6 #Pa s
air_kinematic_viscosity = 19.86e-6 #m2/s
air_specific_heat = 1009 # J/(kg K) at 70°C
air_thermal_conductivity = 0.02952 # W/(m K) at 70°C
air_density = 1.029 # kg/m3 at 70°C 
#Adimensional numbers
ReD = [vel * bed_diameter_m / air_kinematic_viscosity for vel in velocities]
Pr = air_dynamic_viscosity * air_specific_heat / air_thermal_conductivity
Nu = [0.664*Re**0.5*Pr**(1/3) for Re in ReD]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("ReD:", ReD)
print("Pr:", Pr)
print("Nu:", Nu)


ReD: [553.8771399798591, 1208.4592145015106]
Pr: 0.6986436314363144
Nu: [13.866282206670709, 20.481856213911477]


In [3]:
def parse_drying_workbook(
    sheets: dict[str, pd.DataFrame],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Parse repeated sample blocks while retaining source values and flags.

    Sheet names encode air speed in m/s; diameter labels use micrometres.
    Workbook times and masses are minutes and grams, converted to SI in
    the returned tables. Source row numbers remain one-based for auditing.
    """
    experiment_records = []
    observation_records = []
    expected_headers = ["borra +B", "m borra", "u", "adm", "tx"]
    for sheet_name, raw in sheets.items():
        speed_match = re.fullmatch(r"(\d+(?:[.,]\d+)?)ms", sheet_name.strip())
        if speed_match is None:
            raise ValueError(f"Unrecognized air-speed sheet: {sheet_name}")
        speed = float(speed_match.group(1).replace(",", "."))
        if speed <= 0:
            raise ValueError("Air speed must be positive.")
        labels = raw.iloc[:, 0].astype(str).str.strip()
        header_rows = raw.index[labels.eq("Tempo")].tolist()
        dry_rows = raw.index[labels.eq("Mseca")].tolist()
        if len(header_rows) != 1 or len(dry_rows) != 1:
            raise ValueError(
                f"Expected one Tempo and one Mseca row in {sheet_name}."
            )
        header_row, dry_row = header_rows[0], dry_rows[0]
        if header_row < 4 or dry_row <= header_row:
            raise ValueError(
                f"Invalid header/dry-mass layout in {sheet_name}."
            )
        numeric_time = pd.to_numeric(raw.iloc[:, 0], errors="coerce")
        time_rows = raw.index[
            (raw.index > header_row)
            & (raw.index < dry_row)
            & numeric_time.notna()
        ]
        times_min = numeric_time.loc[time_rows].to_numpy(dtype=float)
        if (
            not len(times_min)
            or not np.isfinite(times_min).all()
            or (times_min < 0).any()
        ):
            raise ValueError(f"Invalid measurement times in {sheet_name}.")
        if (
            len(np.unique(times_min)) != len(times_min)
            or not (np.diff(times_min) > 0).all()
        ):
            raise ValueError(
                f"Times must be unique and increasing in {sheet_name}."
            )
        # A diameter heading carries forward across its replicate blocks.
        diameter_m = None
        for start in range(1, raw.shape[1], 5):
            if start + 4 >= raw.shape[1]:
                raise ValueError(f"Incomplete sample block in {sheet_name}.")
            headers = (
                raw.iloc[header_row, start:start + 5]
                .astype(str)
                .str.strip()
                .tolist()
            )
            if headers != expected_headers:
                raise ValueError(
                    f"Unexpected sample headers in {sheet_name}: {headers}"
                )
            diameter_label = raw.iloc[header_row - 4, start]
            if pd.notna(diameter_label):
                match = re.fullmatch(
                    r"(\d+(?:[.,]\d+)?)\s*(?:um|µm|μm)",
                    str(diameter_label).strip(),
                )
                if match is None:
                    raise ValueError(
                        f"Invalid diameter label: {diameter_label}"
                    )
                diameter_m = float(match.group(1).replace(",", ".")) * 1e-6
            if diameter_m is None or diameter_m <= 0:
                raise ValueError("Each sample must have a positive diameter.")
            replicate_value = float(raw.iloc[header_row - 3, start])
            if (
                not np.isfinite(replicate_value)
                or replicate_value < 1
                or not replicate_value.is_integer()
            ):
                raise ValueError("Replicate must be a positive integer.")
            replicate = int(replicate_value)
            tare_g = float(raw.iloc[header_row - 2, start])
            dry_g = float(raw.iloc[dry_row, start + 1])
            values = raw.loc[time_rows, raw.columns[start:start + 5]].apply(
                pd.to_numeric, errors="coerce"
            )
            values.columns = [
                "total_mass_g",
                "sample_mass_g",
                "u_source",
                "adm_source",
                "tx_source",
            ]
            initial_positions = np.flatnonzero(times_min == 0)
            if len(initial_positions) != 1:
                raise ValueError("Every curve needs a measurement at t=0.")
            initial_g = float(
                values.iloc[initial_positions[0]]["sample_mass_g"]
            )
            if (
                not np.isfinite([tare_g, dry_g, initial_g]).all()
                or tare_g < 0
                or not 0 < dry_g < initial_g
            ):
                raise ValueError(
                    f"Invalid tare, dry or initial mass in {sheet_name}, "
                    f"block {start}."
                )
            # u uses initial wet sample mass, not current or dry mass.
            # Dividing by initial_u then gives the normalized moisture adm.
            initial_u = (initial_g - dry_g) / initial_g
            condition_id = f"v{speed:g}_d{diameter_m * 1e6:g}"
            experiment_id = f"{condition_id}_r{replicate}"
            experiment_records.append(
                {
                    "experiment_id": experiment_id,
                    "condition_id": condition_id,
                    "replicate": replicate,
                    "particle_diameter_m": diameter_m,
                    "air_speed_m_s": speed,
                    "bed_depth_m": bed_depth_m,
                    "air_temperature_K": T_air_K,
                    "tare_mass_kg": tare_g / 1000,
                    "dry_mass_kg": dry_g / 1000,
                    "initial_sample_mass_kg": initial_g / 1000,
                    "initial_u": initial_u,
                    "ms_source_g": raw.iloc[header_row - 1, start],
                    "source_sheet": sheet_name,
                }
            )
            previous_mass = None
            for position, (source_index, row) in enumerate(values.iterrows()):
                mass_g = float(row["sample_mass_g"])
                total_g = float(row["total_mass_g"])
                valid_mass = np.isfinite(mass_g) and mass_g >= dry_g
                u = (
                    (mass_g - dry_g) / initial_g
                    if np.isfinite(mass_g)
                    else np.nan
                )
                adm = u / initial_u
                # Flags are diagnostic, not blanket exclusions: mass increases
                # and source-formula mismatches remain usable observations.
                flags = []
                if not np.isfinite(mass_g):
                    flags.append("missing_mass")
                elif mass_g < dry_g:
                    flags.append("below_dry_mass")
                if np.isfinite(mass_g):
                    if (
                        previous_mass is not None
                        and mass_g > previous_mass + 1e-8
                    ):
                        flags.append("mass_increase")
                    previous_mass = mass_g
                if (
                    np.isfinite(total_g)
                    and np.isfinite(mass_g)
                    and not np.isclose(
                        total_g - tare_g, mass_g, atol=1e-6, rtol=0
                    )
                ):
                    flags.append("tare_mismatch")
                for source_column, computed in [
                    ("u_source", u),
                    ("adm_source", adm),
                ]:
                    source_value = float(row[source_column])
                    if (
                        np.isfinite(source_value)
                        and np.isfinite(computed)
                        and not np.isclose(
                            source_value, computed, atol=1e-6, rtol=1e-5
                        )
                    ):
                        flags.append(f"{source_column}_mismatch")
                time_s = times_min[position] * 60
                # Workbook tx is adm/time_min, not a moisture derivative.
                legacy_tx = (
                    adm / times_min[position]
                    if times_min[position] > 0
                    else np.nan
                )
                observation_records.append(
                    {
                        "experiment_id": experiment_id,
                        "time_s": time_s,
                        "total_mass_kg": total_g / 1000,
                        "sample_mass_kg": mass_g / 1000,
                        "u_source": row["u_source"],
                        "adm_source": row["adm_source"],
                        "tx_source": row["tx_source"],
                        "u_recomputed": u,
                        "adm_recomputed": adm,
                        "legacy_adm_per_min": legacy_tx,
                        # Missing/sub-dry masses and tare errors block targets.
                        "target_available": bool(
                            valid_mass and "tare_mismatch" not in flags
                        ),
                        "quality_flags": ";".join(flags),
                        "source_row": int(source_index) + 1,
                    }
                )
    experiments = pd.DataFrame(experiment_records)
    observations = pd.DataFrame(observation_records)
    if experiments.empty or experiments["experiment_id"].duplicated().any():
        raise ValueError("Experiment IDs must be nonempty and unique.")
    if observations.duplicated(["experiment_id", "time_s"]).any():
        raise ValueError("Observation keys must be unique.")
    return experiments, observations

def build_scheduled_intervals(observations: pd.DataFrame) -> pd.DataFrame:
    """Calculate signed moisture-ratio rates only over adjacent valid samples.

    Negative rates are flagged as unavailable for rate targets, while zero
    rates are retained. Invalid scheduled observations break intervals. The
    workbook's legacy ``tx`` field is intentionally not used for this measure.
    """
    required = {
        "experiment_id",
        "time_s",
        "adm_recomputed",
        "target_available",
    }
    if not required.issubset(observations.columns):
        missing = sorted(required - set(observations.columns))
        raise ValueError(f"Observations missing columns: {missing}")

    interval_records = []
    for experiment_id, scheduled in observations.groupby("experiment_id", sort=True):
        ordered = scheduled.sort_values("time_s", kind="stable").reset_index(drop=True)
        times = pd.to_numeric(ordered["time_s"], errors="coerce").to_numpy(dtype=float)
        if not np.isfinite(times).all() or (np.diff(times) <= 0).any():
            raise ValueError(f"Scheduled times must increase for {experiment_id}.")
        moisture_ratio = pd.to_numeric(
            ordered["adm_recomputed"], errors="coerce"
        ).to_numpy(dtype=float)
        available = ordered["target_available"].fillna(False).to_numpy(dtype=bool)
        available &= np.isfinite(moisture_ratio)

        # Only neighboring scheduled rows can form an interval. An unavailable
        # row at either endpoint therefore blocks both neighboring secants.
        for left_index in range(len(ordered) - 1):
            right_index = left_index + 1
            if not (available[left_index] and available[right_index]):
                continue
            interval_scale = total_time
            if not np.isfinite(interval_scale) or interval_scale <= 0:
                raise ValueError("total_time must be finite and positive.")
            dt_star = (times[right_index] - times[left_index]) / interval_scale
            rate_star = -(
                moisture_ratio[right_index] - moisture_ratio[left_index]
            ) / dt_star
            left = ordered.iloc[left_index]
            right = ordered.iloc[right_index]
            interval_records.append(
                {
                    "experiment_id": str(experiment_id),
                    "left_time_s": float(times[left_index]),
                    "right_time_s": float(times[right_index]),
                    "dt_star": float(dt_star),
                    "left_mr": float(moisture_ratio[left_index]),
                    "right_mr": float(moisture_ratio[right_index]),
                    "rate_star": float(rate_star),
                    "left_source_row": int(left.get("source_row", -1)),
                    "right_source_row": int(right.get("source_row", -1)),
                    "right_quality_flags": str(right.get("quality_flags", "")),
                }
            )

    columns = [
        "experiment_id", "left_time_s", "right_time_s", "dt_star",
        "left_mr", "right_mr", "rate_star", "left_source_row",
        "right_source_row", "right_quality_flags",
    ]
    intervals = pd.DataFrame(interval_records, columns=columns)
    if not intervals.empty and not np.isfinite(
        intervals["rate_star"].to_numpy(dtype=float)
    ).all():
        raise ValueError("Measured drying-rate intervals must be finite.")
    intervals["rate_target_available"] = intervals["rate_star"].ge(0)
    return intervals


# The workbook is read-only input: the notebook does not write or alter it.
WORKBOOK_PATH = Path("C:\\Pasta_do_Jota\\TCC\\Estufa convec\u00e7\u00e3o - Borra de caf\u00e9.xlsx")
raw_sheets = pd.read_excel(WORKBOOK_PATH, sheet_name=None, header=None)
experiments, observations = parse_drying_workbook(raw_sheets)
scheduled_intervals = build_scheduled_intervals(observations)
rate_intervals = scheduled_intervals.loc[
    scheduled_intervals["rate_target_available"]
].copy()
negative_rate_intervals = scheduled_intervals.loc[
    ~scheduled_intervals["rate_target_available"]
].copy()
print(
    f"Parsed {len(experiments)} experiments and "
    f"{len(observations)} scheduled observations; "
    f"{observations['target_available'].sum()} usable moisture targets."
)
print(
    f"Retained {len(rate_intervals)} nonnegative rate intervals; "
    f"flagged {len(negative_rate_intervals)} negative rate intervals."
)
if not negative_rate_intervals.empty:
    print("Negative rate intervals excluded from rate targets:")
    print(negative_rate_intervals.to_string(index=False))
print(
    observations.loc[observations["quality_flags"].ne(""), "quality_flags"]
    .value_counts()
    .to_string()
)

observations.head()

Parsed 24 experiments and 576 scheduled observations; 560 usable moisture targets.
Retained 513 nonnegative rate intervals; flagged 21 negative rate intervals.
Negative rate intervals excluded from rate targets:
experiment_id  left_time_s  right_time_s  dt_star  left_mr  right_mr  rate_star  left_source_row  right_source_row right_quality_flags  rate_target_available
v0.22_d213_r1          0.0          60.0 0.004167 1.000000  1.011230  -2.695289                7                 8       mass_increase                  False
v0.22_d610_r1         60.0         120.0 0.004167 1.000000  1.000024  -0.005725                8                 9       mass_increase                  False
v0.48_d213_r1          0.0          60.0 0.004167 1.000000  1.005323  -1.277434                7                 8       mass_increase                  False
v0.48_d213_r1        180.0         240.0 0.004167 0.962601  0.981523  -4.541362               10                11       mass_increase                  Fals

,experiment_id,time_s,total_mass_kg,sample_mass_kg,u_source,adm_source,tx_source,u_recomputed,adm_recomputed,legacy_adm_per_min,target_available,quality_flags,source_row
0,v0.22_d610_r1,0.0,0.022331,0.008583,0.488442,1.000000,NaN,0.488442,1.000000,NaN,True,,7
1,v0.22_d610_r1,60.0,0.022331,0.008583,0.488442,1.000000,1.000000,0.488442,1.000000,1.000000,True,,8
2,v0.22_d610_r1,120.0,0.022331,0.008583,0.488454,1.000024,0.500012,0.488454,1.000024,0.500012,True,mass_increase,9
3,v0.22_d610_r1,180.0,0.022319,0.008570,0.487009,0.997066,0.332355,0.487009,0.997066,0.332355,True,,10
4,v0.22_d610_r1,240.0,0.022274,0.008526,0.481859,0.986523,0.246631,0.481859,0.986523,0.246631,True,,11


## Strategy C inverse PINN (appended integration)

Run the existing workbook parser cell first. Then run the setup cell below and explicitly execute the final invocation cell to perform the locked, seeded 50-epoch Adam followed by bounded L-BFGS run. The workbook is read-only; results are created only in a new timestamped directory beneath `artifacts/strategy_c_inverse/`.

In [4]:
# Resolve the source package whether Jupyter starts in the repository root or notebooks/.
_strategy_c_roots = [Path.cwd(), *Path.cwd().parents]
STRATEGY_C_REPOSITORY_ROOT = next(
    (candidate for candidate in _strategy_c_roots if (candidate / "src" / "strategy_c").is_dir()),
    None,
)
if STRATEGY_C_REPOSITORY_ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks/ directory.")
if str(STRATEGY_C_REPOSITORY_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(STRATEGY_C_REPOSITORY_ROOT / "src"))

from strategy_c.data import Scales, fit_feature_transform, prepare_curve_data, split_experiments
from strategy_c.physics import PhysicsConfig
from strategy_c.reporting import evaluate_locked_model, export_results
from strategy_c.training import SamplingConfig, TrainingConfig, train_sequentially


def run_strategy_c_inverse_pinn():
    """Prepare measured curves, fit once, then evaluate and export the locked split."""
    # The existing parser is the single source of workbook tables; no second
    # parser or workbook write is performed in this integration.
    if "experiments" not in globals() or "observations" not in globals():
        raise RuntimeError("Run the existing workbook parser cell before this cell.")

    physics_config = PhysicsConfig()
    scales = Scales(
        bed_depth_m=physics_config.bed_depth_m,
        dish_diameter_m=physics_config.dish_diameter_m,
        time_scale_s=physics_config.time_scale_s,
        initial_temperature_K=physics_config.initial_temperature_K,
        temperature_difference_K=physics_config.temperature_difference_K,
    )
    prepared_data = prepare_curve_data(experiments, observations, scales)
    locked_split = split_experiments(prepared_data.experiments, seed=42)
    train_feature_transform = fit_feature_transform(prepared_data, locked_split)

    # Keep one bounded sequential optimization: approved 50-step Adam, then
    # L-BFGS on the exact same model parameters and fixed sampled collocation.
    sampling_config = SamplingConfig(
        domain_points=64,
        initial_points=16,
        boundary_points=16,
        seed=42,
    )
    training_config = TrainingConfig(
        adam_epochs=50,
        adam_learning_rate=0.001,
        lbfgs_max_iter=20,
        lbfgs_max_eval=40,
        lbfgs_history_size=50,
        seed=42,
        device="auto",
        dtype=torch.float64,
        quadrature_points=32,
    )
    training_result = train_sequentially(
        None,
        prepared_data,
        locked_split,
        train_feature_transform,
        sampling_config=sampling_config,
        training_config=training_config,
        physics_config=physics_config,
    )

    # Held-out observations are first consumed here, after training has returned.
    evaluation_result = evaluate_locked_model(
        training_result,
        prepared_data,
        locked_split,
        train_feature_transform,
        physics_config,
        quadrature_points=64,
    )
    output_directory = export_results(
        evaluation_result,
        training_result,
        prepared_data,
        STRATEGY_C_REPOSITORY_ROOT / "artifacts" / "strategy_c_inverse",
        physics_config,
        training_config,
        sampling_config,
        feature_transform=train_feature_transform,
    )

    print(f"Training device: {training_result.provenance.device}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"Training curve IDs: {list(training_result.provenance.train_ids)}")
    print(f"Locked held-out IDs: {list(training_result.provenance.held_out_ids)}")
    print(f"Held-out normalized metrics: {dict(evaluation_result.aggregate_metrics)}")
    print(
        "Learned reference diffusivity (m^2/s): "
        f"{float(training_result.model.reference_diffusivity_m2_s.detach().cpu()):.8g}"
    )
    print(
        "Learned activation energy (J/mol): "
        f"{float(training_result.model.activation_energy_j_mol.detach().cpu()):.8g}"
    )
    for stage_name, history in training_result.stage_history.items():
        print(
            f"{stage_name}: elapsed={history.elapsed_seconds:.3f}s, "
            f"epochs={history.epochs}, iterations={history.iterations}, "
            f"closure_calls={history.closure_calls}"
        )
    print(f"Exclusive run artifacts: {output_directory}")
    print("The smoke run and bounded optimizer settings do not establish convergence.")
    return {
        "data": prepared_data,
        "split": locked_split,
        "feature_transform": train_feature_transform,
        "training": training_result,
        "evaluation": evaluation_result,
        "output_directory": output_directory,
    }

In [5]:
# Explicitly invoke the complete, single sequential Strategy C run.
strategy_c_run = run_strategy_c_inverse_pinn()

Training device: cuda
CUDA available: True
Training curve IDs: ['v0.22_d213_r2', 'v0.22_d213_r3', 'v0.22_d236_r1', 'v0.22_d236_r2', 'v0.22_d450_r1', 'v0.22_d450_r3', 'v0.22_d610_r1', 'v0.22_d610_r3', 'v0.48_d213_r1', 'v0.48_d213_r3', 'v0.48_d236_r1', 'v0.48_d236_r2', 'v0.48_d450_r2', 'v0.48_d450_r3', 'v0.48_d610_r1', 'v0.48_d610_r2']
Locked held-out IDs: ['v0.22_d213_r1', 'v0.22_d236_r3', 'v0.22_d450_r2', 'v0.22_d610_r2', 'v0.48_d213_r2', 'v0.48_d236_r3', 'v0.48_d450_r1', 'v0.48_d610_r3']
Held-out normalized metrics: {'curve_count': 8, 'observation_count': 188, 'equal_curve_mae': 0.2826129872279919, 'equal_curve_rmse': 0.3525268859405273, 'pooled_mae': 0.2824458333077503, 'pooled_rmse': 0.35254873897594163}
Learned reference diffusivity (m^2/s): 9.4837214e-11
Learned activation energy (J/mol): 31363.481
adam: elapsed=6.679s, epochs=50, iterations=0, closure_calls=0
lbfgs: elapsed=3.017s, epochs=0, iterations=20, closure_calls=23
Exclusive run artifacts: c:\Pasta_do_Jota\TCC\Modelling-c